MAPBIOMAS

- Dados mapbiomas no GEE: https://developers.google.com/earth-engine/datasets/publisher/mapbiomas-public
- Dados MapBiomas: https://brasil.mapbiomas.org/mapbiomas-cobertura-10m/
- GitHub do MapBiomas: https://github.com/mapbiomas-brazil/user-toolkit
- Legenda Mapbiomas: https://brasil.mapbiomas.org/codigos-de-legenda/ e
https://github.com/mapbiomas-brazil/user-toolkit/tree/master/legend-colors

# Inicializando GEE

In [ ]:
# importa bibliotecas
import ee, geemap

# inicializando pelo GEEMAP
geemap.ee_initialize(project='ee-enrique')

# Mapas do Mapbiomas

Carregando paleta de cores do MapBiomas

In [ ]:
# parâmetros de visualização
param_vis = {'min': 0,
             'max': 75,
             'palette': ['ffffff', '32a65e', '32a65e', '1f8d49', '7dc975',  # 0-4
                         '04381d', '026975', '000000', '000000', '7a6c00',  # 5-9
                         'ad975a', '519799', 'd6bc74', 'd89f5c', 'FFFFB2',  # 10-14
                         'edde8e', '000000', '000000', 'f5b3c8', 'C27BA0',  # 15-19
                         'db7093', 'ffefc3', 'db4d4f', 'ffa07a', 'd4271e',  # 20-24
                         'db4d4f', '0000FF', '000000', '000000', 'ffaa5f',  # 25-29
                         '9c0027', '091077', 'fc8114', '2532e4', '93dfe6',  # 30-34
                         '9065d0', 'd082de', '000000', '000000', 'f5b3c8',  # 35-39
                         'c71585', 'f54ca9', 'cca0d4', 'dbd26b', '807a40',  # 40-44
                         'e04cfa', 'd68fe2', '9932cc', 'e6ccff', '02d659',  # 45-49
                         'ad5100', '000000', '000000', '000000', '000000',  # 50-54
                         '000000', '000000', 'CC66FF', 'FF6666', '006400',  # 55-59
                         '8d9e8b', 'f5d5d5', 'ff69b4', 'ebf8b5', '000000',  # 60-64
                         '000000', '91ff36', '7dc975', 'e97a7a', '0fffe3',  # 65-69
                         '000000', '000000', '000000', '000000', '000000',  # 70-74
                         'c12100'  # 75
                          ]
              }

# legenda personalizada do mapbiomas
legenda_mapbiomas = {'3 - Formação Florestal': '1f8d49',
                     '4 - Formação Savânica': '7dc975',
                     '5 - Mangue': '04381d',
                     '6 - Floresta Alagável': '026975',
                     '9 - Floresta Plantada': '7a6c00',
                     '11 - Área Úmida': '519799',
                     '12 - Formação Campestre': 'd6bc74',
                     '13 - Outras Formações': 'd89f5c',
                     '15 - Pastagem': 'edde8e',
                     '18 - Agricultura': 'f5b3c8',
                     '20 - Cana-de-açúcar': 'db7093',
                     '21 - Mosaico de Usos': 'ffefc3',
                     '22 - Área não Vegetada': 'db4d4f',
                     '23 - Praia/Duna': 'ffa07a',
                     '24 - Área Urbana': 'd4271e',
                     '25 - Outras Áreas': 'db4d4f',
                     '29 - Afloramento Rochoso': 'ffaa5f',
                     '30 - Mineração': '9c0027',
                     '31 - Aquicultura': '091077',
                     '32 - Apicum': 'fc8114',
                     '33 - Rios/Lagos/Oceano': '2532e4',
                     '34 - Glaciar': '93dfe6',
                     '39 - Soja': 'f5b3c8',
                     '40 - Arroz': 'c71585',
                     '41 - Outras Lavouras': 'f54ca9',
                     '46 - Café': 'd68fe2',
                     '47 - Citros': '9932cc',
                     '48 - Outras Perenes': 'e6ccff',
                     '49 - Vegetação em Dunas': '02d659',
                     '50 - Vegetação em Dunas Herbácea': 'ad5100',
                     '61 - Salina': 'f5d5d5',
                     '62 - Algodão': 'ff69b4',
                     '69 - Recifes de Coral': '0fffe3',
                     '75 - Usina Solar': 'c12100'}

# extrai listas para a função add_legend
legenda_keys = list(legenda_mapbiomas.keys())
legenda_colors = ['#' + cor for cor in legenda_mapbiomas.values()]

# mostra os dados de nomes e cores
print(legenda_keys)
print(legenda_colors)

Carregando dados

In [ ]:
mapbiomas = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1')
mapbiomas

In [ ]:
mapbiomas.first().bandNames().getInfo()

Plotando mapas em camadas

In [ ]:
# região
regiao = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# carrega dataset
mapbiomas = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1') \
              .filter(ee.Filter.eq('collection_id', 10.0)) \
              .filter(ee.Filter.eq('version', 'v1'))

# seleciona 2020 e 2024
mapa_1990 = mapbiomas.filter(ee.Filter.eq('year', 1990)).first().clip(regiao)
mapa_2024 = mapbiomas.filter(ee.Filter.eq('year', 2024)).first().clip(regiao)

# cria a moldura do mapa
Map = geemap.Map()

# centraliza o mapa
Map.centerObject(regiao, 12)

# layers
Map.addLayer(mapa_1990, param_vis, '1990')
Map.addLayer(mapa_2024, param_vis, '2024')

# legenda
Map.add_legend(title="MapBiomas v10 - Classes",
               legend_dict=legenda_mapbiomas,
               position='bottomright',
               draggable=True)

# exibe na tela
Map

Plotando com Split map

In [ ]:
# região
regiao = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# carrega dataset
mapbiomas = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1') \
              .filter(ee.Filter.eq('collection_id', 10.0)) \
              .filter(ee.Filter.eq('version', 'v1'))

# seleciona 2020 e 2024
mapa_1990 = mapbiomas.filter(ee.Filter.eq('year', 1990)).first().clip(regiao)
mapa_2024 = mapbiomas.filter(ee.Filter.eq('year', 2024)).first().clip(regiao)

# cria a moldura do mapa
Map = geemap.Map()

# centraliza o mapa
Map.centerObject(regiao, 12)

# split map
Map.split_map(left_layer=geemap.ee_tile_layer(mapa_1990, param_vis, 'Mapbiomas_2016'),
              right_layer=geemap.ee_tile_layer(mapa_2024, param_vis, 'Mapbiomas_2022'))

# exibe na tela
Map

Plotando com dois paineis

In [ ]:
# região
regiao = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
regiao = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# carrega dataset
mapbiomas = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1') \
              .filter(ee.Filter.eq('collection_id', 10.0)) \
              .filter(ee.Filter.eq('version', 'v1'))

# seleciona 2020 e 2024
mapa_1990 = mapbiomas.filter(ee.Filter.eq('year', 1990)).first().clip(regiao)
mapa_2024 = mapbiomas.filter(ee.Filter.eq('year', 2024)).first().clip(regiao)

# monta painel com 4 mapas
geemap.linked_maps(rows = 1,
                   cols = 2,
                   height = "600px",
                   center = [-22.42, -45.4],
                   zoom = 12,
                   ee_objects = [mapa_1990, mapa_2024],
                   vis_params = [param_vis, param_vis],
                   labels = ['1990', '2024'],
                   label_position = "topright")

Salvando imagem no drive

In [ ]:
# salvando imagem tif
geemap.ee_export_image(mapa_2024,
                       filename='mapbiomas_itajuba_2024.tif',
                       scale=30,
                       region=regiao.geometry(),
                       file_per_band=False)

Instala e importa biblioteca para ler arquivo TIF

In [ ]:
# instala e importa biblioteca para ler arquivo TIF
!pip install rioxarray -q
import rioxarray

In [ ]:
# leitura do arquivo de relevo
ds = rioxarray.open_rasterio('/content/mapbiomas_itajuba_2024.tif')
ds

In [ ]:
# plota figura de um ano
ds[0,:,:].plot()

# Calculando a área

In [87]:
# define a região
regiao = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# carrega os dados
mapbiomas = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1') \
              .filter(ee.Filter.eq('collection_id', 10.0)) \
              .filter(ee.Filter.eq('version', 'v1')) \
              .map(lambda img: img.clip(regiao).copyProperties(img, ['system:time_start', 'year']))

# seleciona 2020 e 2024
mapa_1990 = mapbiomas.filter(ee.Filter.eq('year', 1990)).first()
mapa_2024 = mapbiomas.filter(ee.Filter.eq('year', 2024)).first()

# calcula área de cada classe em km²
areas = geemap.image_area_by_group(mapa_2024,        # Imagem classificada (MapBiomas) para 2024
                                   region=regiao,    # Região de análise (geometria)
                                   scale=30,         # Resolução espacial (30m = 900m²/pixel)
                                   denominator=1e6,  # Divisor para converter unidades (1e6 = m² → km²)
                                   decimal_places=4, # Casas decimais no resultado
                                   verbose=True      # Mostra progresso no console
                                   )

# imprime resultados
print("\n 🌳 DISTRIBUIÇÃO DE ÁREA - ITAJUBÁ")
for classe, area_km2_value in areas['area'].items():
    print(f"Classe {classe}: {area_km2_value:,.2f} km²")

Calculating area for group 3 ...
Calculating area for group 9 ...
Calculating area for group 11 ...
Calculating area for group 12 ...
Calculating area for group 15 ...
Calculating area for group 20 ...
Calculating area for group 21 ...
Calculating area for group 24 ...
Calculating area for group 25 ...
Calculating area for group 33 ...
Calculating area for group 41 ...
Calculating area for group 46 ...
Calculating area for group 48 ...

 🌳 DISTRIBUIÇÃO DE ÁREA - ITAJUBÁ
Classe 3: 70.21 km²
Classe 9: 1.40 km²
Classe 11: 0.04 km²
Classe 12: 0.40 km²
Classe 15: 138.78 km²
Classe 20: 0.01 km²
Classe 21: 45.29 km²
Classe 24: 16.39 km²
Classe 25: 0.16 km²
Classe 33: 0.12 km²
Classe 41: 15.96 km²
Classe 46: 1.76 km²
Classe 48: 0.01 km²
